# Vegetation Phenology Analysis for Mission Planning

Vegetation phenology — the seasonal cycle of leaf-out, peak greenness,
and senescence — is a key factor when planning airborne imaging campaigns.
Collecting data at the wrong phenological stage can compromise science
objectives (e.g. mapping species composition requires full canopy, while
forest structure surveys may prefer leaf-off conditions).

HyPlan's `phenology` module retrieves historical MODIS vegetation data
from NASA EarthData to help answer:

- **When is vegetation at peak greenness?**
- **How variable is phenology timing year-to-year?**
- **What time windows combine clear skies AND desired vegetation stage?**

## Supported MODIS Products

| Product | Short Name | Resolution | Temporal | Variables |
|---------|-----------|-----------|----------|-----------|
| Vegetation Indices | MOD13A1 / MYD13A1 | 500 m | 16-day | NDVI, EVI |
| Leaf Area Index | MOD15A2H | 500 m | 8-day | LAI, FPAR |
| Phenology Transitions | MCD12Q2 | 500 m | Yearly | Greenup, peak, senescence dates |

## This Notebook

1. Study area definition
2. Fetching phenology data from EarthData (requires credentials)
3. Seasonal profile analysis (typical-year NDVI/EVI curves)
4. Phenological stage calendar (greenup → dormancy timeline)
5. Year-over-year variability heatmap
6. Combined cloud + phenology analysis for optimal collection windows

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from hyplan.phenology import (
    fetch_phenology,
    summarize_phenology_by_doy,
    extract_phenology_stages,
    plot_seasonal_profile,
    plot_phenology_calendar,
    plot_year_over_year_heatmap,
    plot_cloud_phenology_combined,
)

## 1. Study Area Definition

We use the same Western US study areas as the cloud analysis notebook.
Each polygon represents a flight box that needs to be imaged.

In [ ]:
polygon_file = "exampledata/wdts.geojson"

gdf = gpd.read_file(polygon_file)
print(f"Study areas: {len(gdf)} polygons")
print(gdf[["Name"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 6))
gdf.plot(ax=ax, alpha=0.4, edgecolor="black")
for _, row in gdf.iterrows():
    centroid = row.geometry.centroid
    ax.text(centroid.x, centroid.y, row["Name"], fontsize=8, ha="center", va="center")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Study Area Polygons")
plt.tight_layout()

## 2. Fetching Phenology Data from EarthData

`fetch_phenology` searches NASA's Common Metadata Repository (CMR) for
MODIS granules covering your study areas, downloads them to a local cache,
and extracts QA-filtered vegetation index values per polygon.

**Prerequisites:**
- `pip install hyplan[phenology]`
- NASA Earthdata account ([register here](https://urs.earthdata.nasa.gov))
- Set `EARTHDATA_TOKEN` environment variable, or configure `~/.netrc`

The call below shows the API for fetching 20 years of NDVI data.
Since it requires EarthData credentials and downloads several GB of
MODIS granules, the rest of this notebook uses synthetic data that
mimics realistic phenology patterns for the Western US sites.

In [ ]:
# --- Uncomment to fetch real data (requires EarthData credentials) ---
#
# ndvi_df = fetch_phenology(
#     polygon_file=polygon_file,
#     product="ndvi",
#     year_start=2003,
#     year_stop=2022,
#     satellite="terra",        # "terra", "aqua", or "combined"
#     spatial_mode="mean",      # "mean" or "pixel_stats"
# )
#
# # Phenology transitions (greenup, peak, senescence dates)
# pheno_df = fetch_phenology(
#     polygon_file=polygon_file,
#     product="phenology",
#     year_start=2003,
#     year_stop=2022,
# )

print("fetch_phenology() parameters:")
print("  product:      'ndvi', 'evi', 'lai', 'fpar', 'phenology'")
print("  satellite:    'terra', 'aqua', 'combined' (ndvi/evi only)")
print("  spatial_mode: 'mean' or 'pixel_stats'")
print("  year_start:   default 2003")
print("  year_stop:    default 2022")

### Synthetic Data for Demonstration

The synthetic data below mimics realistic NDVI phenology patterns for
five Western US sites with different vegetation types:

- **santa_barbara** — Coastal chaparral, moderate year-round greenness
- **socal** — Semi-arid scrub, spring green-up after winter rains
- **yosemite** — Mixed conifer forest, strong summer peak
- **tahoe** — High-elevation conifer, snow-limited short growing season
- **bay_area** — Mediterranean grassland, winter-green / summer-brown

In [ ]:
import datetime as dt

np.random.seed(42)

# Site-specific phenology parameters: (base, amplitude, phase_doy, noise_std)
# NDVI ≈ base + amplitude * sin(2π(doy - phase) / 365)
site_params = {
    "santa_barbara": (0.35, 0.15, 90, 0.03),   # mild seasonal cycle
    "socal":         (0.20, 0.15, 75, 0.04),    # spring peak, dry summer
    "yosemite":      (0.30, 0.35, 180, 0.04),   # strong summer peak
    "tahoe":         (0.25, 0.30, 190, 0.05),   # late summer peak, snow winter
    "bay_area":      (0.35, 0.20, 60, 0.03),    # winter-green, peaks early spring
}

years = range(2003, 2023)
doys = range(1, 366, 16)  # 16-day MODIS composites

rows = []
for year in years:
    for doy in doys:
        for site, (base, amp, phase, noise) in site_params.items():
            # Add interannual variability (±10% of amplitude)
            year_offset = np.random.uniform(-0.1, 0.1) * amp
            value = base + (amp + year_offset) * np.sin(
                2 * np.pi * (doy - phase) / 365
            )
            value += np.random.normal(0, noise)
            value = np.clip(value, 0.0, 1.0)

            rows.append({
                "polygon_id": site,
                "date": dt.datetime(year, 1, 1) + dt.timedelta(days=doy - 1),
                "year": year,
                "day_of_year": doy,
                "value": round(value, 4),
            })

ndvi_df = pd.DataFrame(rows)
print(f"Synthetic NDVI data: {len(ndvi_df)} rows")
print(f"Polygons: {sorted(ndvi_df['polygon_id'].unique())}")
print(f"Years: {ndvi_df['year'].min()}–{ndvi_df['year'].max()}")
print(f"NDVI range: {ndvi_df['value'].min():.2f} – {ndvi_df['value'].max():.2f}")
ndvi_df.head(10)

## 3. Seasonal Profile Analysis

`summarize_phenology_by_doy` averages NDVI across all years for each
day-of-year, producing a "typical year" phenology curve per site.
This is analogous to `summarize_cloud_fraction_by_doy` in the clouds module.

The optional `window` parameter applies centered rolling-mean smoothing.

In [ ]:
# Compute DOY summary with 3-point smoothing
ndvi_summary = summarize_phenology_by_doy(ndvi_df, window=3)

print(f"Summary: {len(ndvi_summary)} rows")
print(f"Columns: {list(ndvi_summary.columns)}")
ndvi_summary.head(10)

In [ ]:
# Plot seasonal NDVI profiles for all sites
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

plot_seasonal_profile(ndvi_summary, ax=axes[0], ylabel="NDVI")
axes[0].set_title("Seasonal NDVI Profile (2003–2022 mean ± 1σ)")
axes[0].set_ylim(0, 0.8)

# Without std bands for a cleaner comparison
plot_seasonal_profile(ndvi_summary, ax=axes[1], show_std=False, ylabel="NDVI")
axes[1].set_title("Mean Only (3-point smoothed)")
axes[1].set_ylim(0, 0.8)

fig.tight_layout()
plt.show()

# Identify peak greenness timing per site
print("\nPeak NDVI timing per site:")
for site in sorted(ndvi_summary["polygon_id"].unique()):
    sub = ndvi_summary[ndvi_summary["polygon_id"] == site].sort_values(
        "value_mean", ascending=False
    )
    peak = sub.iloc[0]
    print(f"  {site:16s}  DOY {peak['day_of_year']:3.0f}  NDVI = {peak['value_mean']:.2f}")

## 4. Phenological Stage Calendar

The MCD12Q2 product provides transition dates (greenup, peak, senescence,
dormancy) for each year. `extract_phenology_stages` computes the multi-year
mean and standard deviation of each transition, and `plot_phenology_calendar`
renders a Gantt-style chart showing when each site is in each phenological
stage.

This visualization makes it easy to pick campaign windows that target a
specific vegetation state (e.g. "peak greenness at all sites").

In [ ]:
# Synthetic MCD12Q2-style phenology transition data
# In practice, this comes from fetch_phenology(product="phenology")

np.random.seed(123)

# Realistic transition DOYs for Western US sites
transition_params = {
    #                   greenup  midgrnup  peak  maturity  midgrndn  senescence  dormancy
    "bay_area":       (  45,       70,     100,    130,      200,       240,       290),
    "santa_barbara":  (  60,       85,     120,    150,      220,       260,       310),
    "socal":          (  50,       75,     105,    135,      195,       235,       285),
    "yosemite":       ( 100,      130,     170,    200,      250,       280,       320),
    "tahoe":          ( 120,      150,     190,    220,      260,       290,       330),
}

pheno_rows = []
for year in range(2003, 2023):
    for site, doys in transition_params.items():
        row = {"polygon_id": site, "year": year}
        stage_names = [
            "greenup_doy", "midgreenup_doy", "peak_doy", "maturity_doy",
            "midgreendown_doy", "senescence_doy", "dormancy_doy",
        ]
        for name, base_doy in zip(stage_names, doys):
            # Add interannual variability (±7 days)
            row[name] = base_doy + np.random.normal(0, 4)
        pheno_rows.append(row)

pheno_df = pd.DataFrame(pheno_rows)
print(f"Phenology transitions: {len(pheno_df)} rows, {len(pheno_df['polygon_id'].unique())} sites")
pheno_df.head()

In [ ]:
# Compute multi-year mean/std of transition dates
stages = extract_phenology_stages(pheno_df)

print("Mean phenological transition DOYs:\n")
print(stages[["polygon_id", "greenup_doy_mean", "peak_doy_mean",
              "senescence_doy_mean", "dormancy_doy_mean"]].to_string(index=False))

In [ ]:
# Gantt chart of phenological stages
ax = plot_phenology_calendar(stages)
ax.set_title("Phenological Stage Calendar (2003–2022 mean, error bars = 1σ)")
plt.tight_layout()
plt.show()

## 5. Year-over-Year Variability

The year-over-year heatmap reveals interannual variability in phenology
timing. Each row is a year, each column is a day-of-year, and the color
represents NDVI. Look for:

- **Consistent timing** — vertical color bands stay in the same position
- **Early/late years** — bands shift left or right
- **Drought years** — overall dimmer colors (lower peak NDVI)

In [ ]:
# Heatmaps for two contrasting sites
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_year_over_year_heatmap(
    ndvi_df, polygon_id="yosemite", ax=axes[0],
    cmap="YlGn", vmin=0.0, vmax=0.7,
)

plot_year_over_year_heatmap(
    ndvi_df, polygon_id="bay_area", ax=axes[1],
    cmap="YlGn", vmin=0.0, vmax=0.7,
)

fig.suptitle("NDVI Year-over-Year Variability", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 6. Combined Cloud + Phenology Analysis

The most powerful planning insight comes from combining cloud fraction
climatology with vegetation phenology. `plot_cloud_phenology_combined`
overlays both datasets to identify time windows where:

1. **Cloud cover is low** (flyable conditions)
2. **Vegetation is at the desired stage** (science objectives met)

Two layout modes are available:
- `"overlay"` — twin y-axes on a single plot
- `"side_by_side"` — two stacked panels with a shared x-axis

In [ ]:
# Generate synthetic cloud fraction summary (matching clouds module output)
# In practice: cloud_summary = summarize_cloud_fraction_by_doy(cloud_df)

np.random.seed(99)
cloud_rows = []
for doy in range(1, 366, 1):
    for site, params in site_params.items():
        # Cloud fraction: higher in winter, lower in summer (California pattern)
        cf = 0.55 - 0.30 * np.sin(2 * np.pi * (doy - 30) / 365)
        cf += np.random.normal(0, 0.02)
        cf = np.clip(cf, 0, 1)
        cloud_rows.append({
            "polygon_id": site,
            "day_of_year": doy,
            "cloud_fraction_mean": round(cf, 3),
            "cloud_fraction_std": round(abs(np.random.normal(0.10, 0.02)), 3),
        })

cloud_summary = pd.DataFrame(cloud_rows)
print(f"Cloud summary: {len(cloud_summary)} rows")

In [ ]:
# Overlay mode: cloud fraction (blue, left axis) + NDVI (green, right axis)
# Filter to one site for clarity
site = "yosemite"
cloud_one = cloud_summary[cloud_summary["polygon_id"] == site]
ndvi_one = ndvi_summary[ndvi_summary["polygon_id"] == site]

ax = plot_cloud_phenology_combined(
    cloud_one, ndvi_one, layout="overlay",
)
ax.set_title(f"Cloud Fraction vs. NDVI Seasonality — {site}")
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side mode: all sites, two stacked panels
fig = plot_cloud_phenology_combined(
    cloud_summary, ndvi_summary, layout="side_by_side",
)
plt.show()

## 7. Planning Recommendations

Combining the phenology and cloud analyses, we can identify optimal
collection windows for each site.

In [ ]:
# Find optimal windows: peak NDVI with low cloud fraction
print("Optimal collection windows (peak greenness + clear skies):\n")
print(f"{'Site':16s}  {'Peak NDVI DOY':>14s}  {'Cloud Frac':>11s}  {'Window':>20s}")
print("-" * 67)

for site in sorted(ndvi_summary["polygon_id"].unique()):
    # Find DOY of peak NDVI
    ndvi_sub = ndvi_summary[ndvi_summary["polygon_id"] == site].sort_values(
        "value_mean", ascending=False,
    )
    peak_doy = int(ndvi_sub.iloc[0]["day_of_year"])

    # Get cloud fraction around peak
    cloud_sub = cloud_summary[cloud_summary["polygon_id"] == site]
    cloud_at_peak = cloud_sub[
        (cloud_sub["day_of_year"] >= peak_doy - 16)
        & (cloud_sub["day_of_year"] <= peak_doy + 16)
    ]["cloud_fraction_mean"].mean()

    # Convert DOY to approximate date
    approx_date = dt.datetime(2020, 1, 1) + dt.timedelta(days=peak_doy - 1)
    window_start = approx_date - dt.timedelta(days=16)
    window_end = approx_date + dt.timedelta(days=16)

    print(
        f"{site:16s}  {peak_doy:14d}  {cloud_at_peak:11.0%}  "
        f"{window_start.strftime('%b %d')} – {window_end.strftime('%b %d')}"
    )

## Summary

The `hyplan.phenology` module provides:

| Function | Purpose |
|----------|---------|
| `fetch_phenology()` | Retrieve NDVI/EVI/LAI/FPAR or phenology transitions from MODIS via EarthData |
| `fetch_phenology_spatial()` | Per-pixel time-averaged vegetation maps |
| `summarize_phenology_by_doy()` | Compute typical-year seasonal profile |
| `extract_phenology_stages()` | Multi-year mean/std of transition dates |
| `plot_seasonal_profile()` | Line plot of VI seasonality |
| `plot_phenology_calendar()` | Gantt chart of phenological stages |
| `plot_year_over_year_heatmap()` | DOY × year heatmap |
| `plot_cloud_phenology_combined()` | Overlay or side-by-side cloud + phenology |

**Installation:** `pip install hyplan[phenology]`

**Credentials:** NASA Earthdata account required — set `EARTHDATA_TOKEN`
or configure `~/.netrc`.  Register at https://urs.earthdata.nasa.gov